# 韩国海关前20日半导体同比：保守官方登记日到美股 Qlib 会话来源图

2018–2026当前原报105个月，严格页面登记日之后首个美股会话才可使用；历史首次公开版本资格仍为false。
首次执行在训练/留出来源列名不一致处停下；修复只规范训练HWP哈希列名，消耗见attempt1_failure.json与repair_plan.json。


In [ ]:
import hashlib,json,os,pathlib,shlex,re
import numpy as np,pandas as pd,qlib
from bs4 import BeautifulSoup
from qlib.data import D
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from qlib.workflow import R
from src.data.nas_market_data import archive_nas_recorder,_settings,_ssh
root=pathlib.Path.cwd();card=root/'research/kcs_early20_2018_2026_qlib_availability_map_20261023';plan=json.loads((card/'plan.json').read_text());src=plan['sources'];sha=lambda p:hashlib.sha256(pathlib.Path(p).read_bytes()).hexdigest();assert plan['frozen'] and not (card/'run_state.json').exists() and json.loads((card/'repair_plan.json').read_text())['status']=='approved_mechanical_schema_alias_only'
train=root/src['training_numeric_card'];held=root/src['held_numeric_card'];train_orig=root/src['training_original_card'];held_orig=root/src['held_original_card'];aug_card=root/src['August_repair_card'];train_art=train/'qlib_recorder'/src['training_numeric_experiment_id']/src['training_numeric_recorder_id']/'artifacts/analysis/monthly_reported_YoY';held_art=held/'qlib_recorder'/src['held_numeric_experiment_id']/src['held_numeric_recorder_id']/'artifacts/analysis/monthly_reported_YoY'
for p,h in [(train/'summary.json',src['training_numeric_summary_sha256']),(train_art,src['training_numeric_artifact_sha256']),(held/'summary.json',src['held_numeric_summary_sha256']),(held_art,src['held_numeric_artifact_sha256']),(train_orig/'summary.json',src['training_original_summary_sha256']),(held_orig/'summary.json',src['held_original_summary_sha256']),(aug_card/'summary.json',src['August_repair_summary_sha256']),(root/src['Qlib_provider_uri']/'qlib_store_manifest.json',src['Qlib_provider_manifest_sha256'])]:assert sha(p)==h,(p,sha(p),h)
assert json.loads((train_orig/'source_restore.json').read_text())['status']=='NAS_five_year_original_groups_all_members_cold_restored_exact';assert json.loads((held_orig/'source_restore.json').read_text())['status']=='NAS_four_held_year_source_groups_all_members_cold_restored_exact';assert json.loads((aug_card/'source_restore.json').read_text())['status']=='NAS_2025_Aug_repair_all_members_cold_restored_exact'
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/src['Qlib_provider_uri']),region='us',kernels=1);cal=pd.DatetimeIndex(D.calendar(start_time='2011-01-03',end_time='2026-10-09',freq='day'));assert len(cal)==plan['cohorts']['full_Stage1_Qlib_calendar'] and cal.is_unique and cal.is_monotonic_increasing
with R.uri_context(str(train/'qlib_recorder')):
 r=R.get_recorder(experiment_id=src['training_numeric_experiment_id'],recorder_id=src['training_numeric_recorder_id']);assert r.status=='FINISHED';training=r.load_object('analysis/monthly_reported_YoY')
with R.uri_context(str(held/'qlib_recorder')):
 r=R.get_recorder(experiment_id=src['held_numeric_experiment_id'],recorder_id=src['held_numeric_recorder_id']);assert r.status=='FINISHED';evaluation=r.load_object('analysis/monthly_reported_YoY')
assert len(training)==60 and len(evaluation)==45;training=training.rename(columns={'HWP_sha256':'source_sha256'}).assign(source_format='hwp');numeric=pd.concat([training[['period','reported_semiconductor_export_yoy_pct','source_sha256','source_format']],evaluation[['period','reported_semiconductor_export_yoy_pct','source_sha256','source_format']]],ignore_index=True);assert len(numeric)==105 and numeric.period.is_unique and numeric.period.is_monotonic_increasing
source_pages=[]
for year in range(2018,2023):
 m=json.loads((root/f'data/snapshots/kcs_early20_2018_2022_original_capture_20261023/{year}/year_manifest.json').read_text());source_pages.extend((x['month'],x) for x in m['months'])
for year in range(2023,2027):
 m=json.loads((root/f'data/snapshots/kcs_early20_2023_2026_held_original_capture_20261023/{year}/year_manifest.json').read_text());source_pages.extend((x['month'],x['page']) for x in m['months'] if x['page'])
repair=json.loads((root/'data/snapshots/kcs_early20_2025_aug_transport_repair_20261023/repair_manifest.json').read_text());source_pages.append(('2025-08',repair['new_page']));assert len(source_pages)==105
page_map={month:entry for month,entry in source_pages};assert len(page_map)==105;event_rows=[]
for row in numeric.itertuples():
 page=page_map[row.period];p=root/page['local_path'];assert sha(p)==page['body_sha256'] and page['http_status']==200;source=BeautifulSoup(p.read_bytes(),'html.parser');tag=source.select_one('.article_head .variety .info span');assert tag is not None;printed=tag.get_text(' ',strip=True);assert re.fullmatch(r'\d{4}\.\d{2}\.\d{2}',printed);date=pd.Timestamp(printed.replace('.','-'));assert date.strftime('%Y-%m')==row.period and 20<=date.day<=27;position=int(cal.searchsorted(date,side='right'));assert position<len(cal);eligible=cal[position];assert eligible>date
 event_rows.append({'period':row.period,'reported_semiconductor_export_yoy_pct':float(row.reported_semiconductor_export_yoy_pct),'source_format':row.source_format,'source_sha256':row.source_sha256,'page_sha256':page['body_sha256'],'portal_registered_date':date,'eligible_trade_date':eligible,'eligible_calendar_position':position,'historical_first_public_byte_vintage_qualified':False})
events=pd.DataFrame(event_rows);assert len(events)==105 and events.eligible_trade_date.is_unique and events.eligible_trade_date.is_monotonic_increasing
loc=np.searchsorted(events.eligible_trade_date.to_numpy(dtype='datetime64[ns]'),cal.to_numpy(dtype='datetime64[ns]'),side='right')-1;known=loc>=0;yoy=np.full(len(cal),np.nan);age=np.full(len(cal),np.nan);period=np.full(len(cal),None,dtype=object);portal=np.full(len(cal),np.datetime64('NaT'),dtype='datetime64[ns]');event=np.zeros(len(cal),dtype=bool);yoy[known]=events.reported_semiconductor_export_yoy_pct.to_numpy(float)[loc[known]];age[known]=np.arange(len(cal))[known]-events.eligible_calendar_position.to_numpy(int)[loc[known]];period[known]=events.period.to_numpy(object)[loc[known]];portal[known]=events.portal_registered_date.to_numpy(dtype='datetime64[ns]')[loc[known]];event[events.eligible_calendar_position.to_numpy(int)]=True
mapped=pd.DataFrame({'trade_date':cal,'kcs_semiconductor_export_yoy_pct':yoy,'source_known':known,'source_period':period,'portal_registered_date':pd.to_datetime(portal),'source_age_sessions':age,'source_event_eligible_today':event,'historical_first_public_byte_vintage_qualified':False});assert len(mapped)==len(cal) and mapped.source_event_eligible_today.sum()==105
coh=plan['cohorts'];tr=mapped.loc[mapped.trade_date.between('2018-01-02','2022-12-30')];he=mapped.loc[mapped.trade_date.between('2023-01-03','2026-10-07')];assert len(tr)==coh['2018_2022_sessions'] and int(tr.source_known.sum())==coh['2018_2022_known'] and int((~tr.source_known).sum())==coh['initial_unknown'] and len(he)==coh['held_2023_2026Oct07_sessions'] and he.source_known.all()
static=pd.concat({'feature':mapped[['kcs_semiconductor_export_yoy_pct']].set_axis(cal),'meta':mapped[['source_known','source_age_sessions','source_event_eligible_today','historical_first_public_byte_vintage_qualified']].set_axis(cal)},axis=1);static.index=pd.MultiIndex.from_arrays([cal,['SOXL_SOXX_CASH']*len(cal)],names=['datetime','instrument']);dataset=DatasetH(handler=DataHandlerLP.from_df(static),segments={'train_2018_2022':('2018-01-02','2022-12-30'),'held':('2023-01-03','2026-10-07'),'full':('2011-01-03','2026-10-09')});assert len(dataset.prepare('train_2018_2022',col_set=['feature','meta']))==1259 and len(dataset.prepare('held',col_set=['feature','meta']))==944
result={'status':'conditional_current_original_KCS_early20_source_Qlib_calendar_complete_held_PIT_vintage_pending','plan_sha256':sha(card/'plan.json'),'source_events':105,'train_events':60,'held_events':45,'full_Qlib_sessions':len(cal),'train_2018_2022_sessions':len(tr),'train_known':int(tr.source_known.sum()),'train_unknown':int((~tr.source_known).sum()),'held_sessions':len(he),'held_known':int(he.source_known.sum()),'first_eligible_trade':str(events.eligible_trade_date.iloc[0].date()),'last_eligible_trade':str(events.eligible_trade_date.iloc[-1].date()),'same_day_portal_to_trade_used':0,'source_historical_first_public_byte_vintage_qualified':False,'working_day_adjusted':False,'SOXX_outcome_reads':0,'model_fits':0,'native_accounts':0};(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
state={'status':'Recorder_started','plan_sha256':result['plan_sha256'],'source_Recorder_loads_cumulative':4,'D_calendar_calls_cumulative':2,'DatasetH_builds':1,'model_fits':0,'native_accounts':0};(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
with R.start(experiment_name=plan['experiment'],uri=str(card/'qlib_recorder')):
 r=R.get_recorder();dataset.config(dump_all=True,recursive=True);r.save_objects(artifact_path='inputs',plan=plan,dataset=dataset);r.save_objects(artifact_path='analysis',source_events=events,mapped_calendar=mapped,result=result);r.log_metrics(events=105,train_known=result['train_known'],held_known=result['held_known'],model_fits=0,native_accounts=0);rid,eid=r.id,str(r.experiment_id)
state['status']='finished';state['recorder_id']=rid;state['experiment_id']=eid;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');evidence=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rid,experiment_id=eid);assert _ssh(_settings(root)['ssh'],'sha256sum '+shlex.quote(evidence['nas_path'])).split()[0]==evidence['sha256']
summary={'experiment':plan['experiment'],'decision':'conditional_KCS_early20_Qlib_map_105_events_held_complete_historical_vintage_unqualified_no_fit','objective_achieved':False,'plan_sha256':result['plan_sha256'],'result':'result.json','recorder_id':rid,'experiment_id':eid,'recorder_uri':str(card/'qlib_recorder'),'NAS_evidence':evidence,'NAS_readback':{'sha256':evidence['sha256'],'verified':True},'source_events':105,'train_known':result['train_known'],'held_known':result['held_known'],'SOXX_outcome_reads':0,'model_fits':0,'native_accounts':0,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/source_events','analysis/mapped_calendar','analysis/result']},'limitations':['Official portal registered date-next-US-session is conservative but not an exact first-public byte certification.','Raw reported YoY is working-day/base sensitive; no adjusted surprise has been constructed.','Qlib source coverage permits only a conditional development information screen, not SOXX promotion.']};(card/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2)+'\n');print(json.dumps({'decision':summary['decision'],'source_events':105,'train_known':result['train_known'],'held_known':result['held_known'],'recorder_id':rid,'NAS_sha256':evidence['sha256']},ensure_ascii=False,indent=2))


## 中文结论

从已NAS冷恢复的2018–2022 **60月**和2023–2026年9月 **45月**原报数值，逐份只读取官方原稿页固定的登记日字段。每月最早允许进入Stage1 09:46决策的时间设为该**登记日之后的首个原生美股Qlib交易会话**，不使用同日收盘或未来月份；105个事件的可用会话互不重叠。以原生 `D.calendar` 和 `DatasetH` 保留3,966个完整日位置：2018–2022的1,259日有 **1,245日**来源已知，初始14日未知；2023–2026固定留出 **944/944日**已知。映射的月度原报同比保持原符号，没有按工作日调整或人为零填补。

首次执行在建DatasetH之前因训练数值卡的 `HWP_sha256` 与留出卡的 `source_sha256` 列名差异停止，实际2次源Recorder读取、1次Qlib日历读取、0新Recorder。已按 [机械修复计划](repair_plan.json) 仅把训练哈希列改名并补字面格式 `hwp`；续跑后累计源读取4次、日历2次，1个DatasetH和1个原生Recorder，没有SOXX结果、拟合或账户。失败收据原样保留。

决定 `conditional_KCS_early20_Qlib_map_105_events_held_complete_historical_vintage_unqualified_no_fit`。原生Recorder `5b3c2e44c6db494cb3015d00648ba162` 的105行事件、3,966日映射和结果对象已NAS定向冷恢复，SHA-256 `617822f6dead0ae64ccb3091d84800508daf481576b365fe5382fbf10db60d8a`。**资格界限仍然关键：**这是以官方页面登记日做的保守时钟；今天可下载HWP/HWPX/PDF的历史首次公开字节及修订版本未独立认证，所以只能授权条件性开发信息检验，不能作可晋升的PIT策略证据。下一固定信息门须以原Stage1 39项、同一成熟标签和相同已知日期作匹配对照；只有必要信息增量通过，才另冻完整0bp Tiered自有资金账户。SOXX稳定超额尚未达成。
